In this notebook we will take a closer look into the Random Forest algorithm. First lets do a Random Forest without any hyperparameter tuning and then we will do a Random Forest with hyperparameter tuning.


In [16]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [17]:

# Load the dataset
df = pd.read_csv('winequality-red-clean.csv')
X = df.drop('quality', axis=1)  # Features
y = df['quality']  # Target (quality)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [18]:

# Initialize a DataFrame to store performance results
performance_df = pd.DataFrame(columns=[
    "Model",
    "Train Accuracy",
    "Test Accuracy",
    "Cross-Validation Accuracy (Mean ± Std)",
    "Bias",
    "Variance"
])

# 1. Random Forest without Hyperparameter Tuning
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

# Predict on training and test data
y_train_pred = rf_model.predict(X_train)
y_test_pred = rf_model.predict(X_test)

# Calculate accuracy for both train and test sets
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

# Cross-validation results (5-fold)
cv_scores = cross_val_score(rf_model, X_train, y_train, cv=5)
cv_mean = cv_scores.mean()
cv_std = cv_scores.std()

# Calculate Bias and Variance
bias = 100 - (train_accuracy * 100)  # Bias as 100 - train accuracy
variance = (train_accuracy - test_accuracy) * 100  # Variance as the difference between train and test accuracy

# Store the results in the performance_df DataFrame
performance_df.loc[len(performance_df.index)] = [
    "Random Forest (No Tuning)",
    f"{train_accuracy * 100:.2f}%",
    f"{test_accuracy * 100:.2f}%",
    f"{cv_mean * 100:.2f}% ± {cv_std * 100:.2f}%",
    f"{bias:.2f}%",
    f"{variance:.2f}%"
]

performance_df


,Model,Train Accuracy,Test Accuracy,Cross-Validation Accuracy (Mean ± Std),Bias,Variance
0,Random Forest (No Tuning),100.00%,68.49%,68.36% ± 3.03%,0.00%,31.51%


In [19]:

# 2. Random Forest with Hyperparameter Tuning
# 2. Random Forest with Hyperparameter Tuning
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'max_features': [ 'sqrt', 'log2'],  # Added max_features parameter
}

# Initialize GridSearchCV
grid_search = GridSearchCV(estimator=RandomForestClassifier(random_state=42),
                           param_grid=param_grid,
                           scoring='accuracy',
                           cv=5,
                           n_jobs=-1)

# Fit the model
grid_search.fit(X_train, y_train)
best_rf_model = grid_search.best_estimator_

# Predict on training and test data
y_train_pred_tuned = best_rf_model.predict(X_train)
y_test_pred_tuned = best_rf_model.predict(X_test)

# Calculate accuracy for both train and test sets
train_accuracy_tuned = accuracy_score(y_train, y_train_pred_tuned)
test_accuracy_tuned = accuracy_score(y_test, y_test_pred_tuned)

# Cross-validation results (5-fold)
cv_scores_tuned = cross_val_score(best_rf_model, X_train, y_train, cv=5)
cv_mean_tuned = cv_scores_tuned.mean()
cv_std_tuned = cv_scores_tuned.std()

# Calculate Bias and Variance
bias_tuned = 100 - (train_accuracy_tuned * 100)  # Bias as 100 - train accuracy
variance_tuned = (train_accuracy_tuned - test_accuracy_tuned) * 100  # Variance as the difference between train and test accuracy

# Store the results in the performance_df DataFrame
performance_df.loc[len(performance_df.index)] = [
    "Random Forest (With Tuning)",
    f"{train_accuracy_tuned * 100:.2f}%",
    f"{test_accuracy_tuned * 100:.2f}%",
    f"{cv_mean_tuned * 100:.2f}% ± {cv_std_tuned * 100:.2f}%",
    f"{bias_tuned:.2f}%",
    f"{variance_tuned:.2f}%"
]

# Display the results table
performance_df


,Model,Train Accuracy,Test Accuracy,Cross-Validation Accuracy (Mean ± Std),Bias,Variance
0,Random Forest (No Tuning),100.00%,68.49%,68.36% ± 3.03%,0.00%,31.51%
1,Random Forest (With Tuning),100.00%,68.84%,68.87% ± 2.48%,0.00%,31.16%


Here we can see that the Random Forest model with hyperparameter tuning has a little bit better test accuracy compared to the Random Forest model without hyperparameter tuning. The bias and variance are also lower for the model with hyperparameter tuning. Both are overfitting the data, but the model with hyperparameter tuning is overfitting less. Let's see what we can do about that.

In [20]:
# 3. Random Forest with Hyperparameter Tuning and Regularization
#  Set up the parameter grid for tuning with regularization
param_grid = {
    'n_estimators': [50, 100],         # Number of trees in the forest
    'max_depth': [None, 10, 20],       # Maximum depth of the tree
    'min_samples_split': [2, 5],       # Minimum number of samples required to split an internal node
    'min_samples_leaf': [1, 2],        # Minimum number of samples required to be at a leaf node
    'max_features': ['log2', 'sqrt']    # Number of features to consider for the best split
}

# Initialize GridSearchCV
grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5)

# Fit the model with hyperparameter tuning
grid_search.fit(X_train, y_train)

# Get the best model from grid search
best_rf_model = grid_search.best_estimator_

# Predict on training data and test data with the best model
y_train_pred_tuning = best_rf_model.predict(X_train)
y_test_pred_tuning = best_rf_model.predict(X_test)

# Accuracy for both train and test sets
train_accuracy_tuning = accuracy_score(y_train, y_train_pred_tuning)
test_accuracy_tuning = accuracy_score(y_test, y_test_pred_tuning)

# Cross-validation results (5-fold) with the best model
cv_scores_tuning = cross_val_score(best_rf_model, X_train, y_train, cv=5)
cv_mean_tuning = cv_scores_tuning.mean()
cv_std_tuning = cv_scores_tuning.std()

# Calculate Bias and Variance
bias_tuning = 100 - (train_accuracy_tuning * 100)  # Bias as 100 - train accuracy
variance_tuning = (train_accuracy_tuning - test_accuracy_tuning) * 100  # Variance

# Store the results in the performance_df DataFrame
performance_df.loc[len(performance_df.index)] = [
    "Random Forest (With Tuning and Regularization)",
    f"{train_accuracy_tuning * 100:.2f}%",
    f"{test_accuracy_tuning * 100:.2f}%",
    f"{cv_mean_tuning * 100:.2f}% ± {cv_std_tuning * 100:.2f}%",
    f"{bias_tuning:.2f}%",
    f"{variance_tuning:.2f}%"
]

# Display the performance results
performance_df

,Model,Train Accuracy,Test Accuracy,Cross-Validation Accuracy (Mean ± Std),Bias,Variance
0,Random Forest (No Tuning),100.00%,68.49%,68.36% ± 3.03%,0.00%,31.51%
1,Random Forest (With Tuning),100.00%,68.84%,68.87% ± 2.48%,0.00%,31.16%
2,Random Forest (With Tuning and Regularization),96.48%,68.49%,68.79% ± 2.91%,3.52%,27.99%


We managed to get the variance down a bit, although it is still quite high. The bias is also lower than before. We can try to reduce the variance even more by reducing the number of features. Let's try that.

In [22]:
from sklearn.feature_selection import SelectKBest, f_classif
from imblearn.pipeline import Pipeline

# 4. Random Forest with Hyperparameter Tuning, Regularization, and Feature Reduction

pipeline = Pipeline([
    ('feature_selection', SelectKBest(score_func=f_classif)),
    ('classifier', RandomForestClassifier(random_state=42, class_weight='balanced'))
])

# Set the hyperparameters for tuning: testing feature counts from 1 to 10
param_grid = {
    'feature_selection__k': list(range(1, 11)),  # Testing all numbers from 1 to 10
    'classifier__n_estimators': [50, 100],  # Number of trees in the forest
    'classifier__max_depth': [None, 10, 20],  # Max depth of the trees
    'classifier__min_samples_split': [2, 5],  # Minimum samples required to split an internal node
    'classifier__min_samples_leaf': [1, 2],  # Minimum samples required to be at a leaf node
}

# Perform Grid Search with cross-validation
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1, refit='accuracy')
grid_search.fit(X_train, y_train)

# Get the best estimator
best_model_tuning = grid_search.best_estimator_

# Predict on training and test sets
y_train_pred_tuning = best_model_tuning.predict(X_train)
y_test_pred_tuning = best_model_tuning.predict(X_test)

# Accuracy for training and testing sets
train_accuracy_tuning = accuracy_score(y_train, y_train_pred_tuning)
test_accuracy_tuning = accuracy_score(y_test, y_test_pred_tuning)

# Cross-validation results
cv_scores_tuning = cross_val_score(best_model_tuning, X_train, y_train, cv=5)
cv_mean_tuning = cv_scores_tuning.mean()
cv_std_tuning = cv_scores_tuning.std()

# Calculate Bias and Variance
bias_tuning = 100 - (train_accuracy_tuning * 100)
variance_tuning = (train_accuracy_tuning - test_accuracy_tuning) * 100

# Store the results
performance_df.loc[len(performance_df.index)] = [
    "Random Forest (Feature Reduction)",
    f"{train_accuracy_tuning * 100:.2f}%",
    f"{test_accuracy_tuning * 100:.2f}%",
    f"{cv_mean_tuning * 100:.2f}% ± {cv_std_tuning * 100:.2f}%",
    f"{bias_tuning:.2f}%",
    f"{variance_tuning:.2f}%"
]

# Display the performance DataFrame
performance_df

Fitting 5 folds for each of 240 candidates, totalling 1200 fits



KeyboardInterrupt



That didnt help much. We can try earlier stopping to reduce the variance. Let's try that.


In [23]:
# 5. Random Forest with Early Stopping
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2'],
    'oob_score': [True]  # Enable OOB scoring
}

# Initialize the model
rf_model = RandomForestClassifier(random_state=42)

# Initialize GridSearchCV for hyperparameter tuning
grid_search = GridSearchCV(rf_model, param_grid, cv=5, scoring='accuracy')

# Fit the model to the training data
grid_search.fit(X_train, y_train)

# Best model parameters
best_params = grid_search.best_params_

# Evaluate the model on the validation set using OOB score
val_accuracy = grid_search.best_estimator_.oob_score_

# Train final model on the full training set with the best parameters
final_model = RandomForestClassifier(**best_params, random_state=42)
final_model.fit(X_train, y_train)

# Evaluate on the test set
y_test_pred = final_model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_test_pred)

# Cross-validation results
cv_scores = grid_search.cv_results_['mean_test_score']
cv_std = grid_search.cv_results_['std_test_score']
cv_mean = cv_scores.mean()
cv_std_mean = cv_std.mean()

# Calculate Bias and Variance
bias = 100 - (val_accuracy * 100)  # Bias as 100 - OOB validation accuracy
variance = (val_accuracy - test_accuracy) * 100  # Variance as the difference between OOB and test accuracy


performance_df.loc[len(performance_df.index)] = [
    "Random Forest (With Tuning and OOB)",
    f"{(grid_search.best_score_ * 100):.2f}%",
    f"{test_accuracy * 100:.2f}%",
    f"{cv_mean * 100:.2f}% ± {cv_std_mean * 100:.2f}%",
    f"{bias:.2f}%",
    f"{variance:.2f}%"
]
performance_df

,Model,Train Accuracy,Test Accuracy,Cross-Validation Accuracy (Mean ± Std),Bias,Variance
0,Random Forest (No Tuning),100.00%,68.49%,68.36% ± 3.03%,0.00%,31.51%
1,Random Forest (With Tuning),100.00%,68.84%,68.87% ± 2.48%,0.00%,31.16%
2,Random Forest (With Tuning and Regularization),96.48%,68.49%,68.79% ± 2.91%,3.52%,27.99%
3,Random Forest (Feature Reduction),99.91%,69.18%,68.70% ± 2.60%,0.09%,30.74%
4,Random Forest (With Tuning and OOB),68.87%,68.84%,66.89% ± 2.53%,29.33%,1.83%


Yay!! We managed to reduce the variance quite a bit and our model is not overfitting like crazy anymore and the testing accuracy didn't drop much. 